# Semana 8 · ¿Qué significa «moverse»?

Este cuaderno acompaña a la sesión de proyecto de la semana 8: calcular velocidades a partir de distancias y tiempos, y simular un movimiento rectilíneo paso a paso.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar.

Tiempo estimado: 30-45 minutos.

## 1. Velocidad = distancia / tiempo

La velocidad media es $v = \Delta x / \Delta t$. Para convertir km/h en m/s se divide entre 3,6 (porque $1\ \text{km/h} = 1000\ \text{m} / 3600\ \text{s}$).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def kmh_a_ms(v): return v / 3.6
def ms_a_kmh(v): return v * 3.6

print(kmh_a_ms(100), ms_a_kmh(465.1))

## 2. La velocidad de tu lugar por la rotación de la Tierra

Un punto del ecuador da una vuelta (40 075 km) en un día sidéreo (86 164 s) y por tanto va a $v = C/T$. A una latitud $\varphi$, el círculo que recorre tiene un radio menor, $R\cos\varphi$, así que la velocidad es $v(\varphi) = v_{\text{ecuador}} \cos\varphi$.

In [ ]:
C_ecuador = 40075.0        # km
T_sidereo = 86164.1        # s (día sidéreo)
v_ecuador = C_ecuador / T_sidereo * 1000          # m/s
print(f"Velocidad en el ecuador: {v_ecuador:.1f} m/s = {ms_a_kmh(v_ecuador):.0f} km/h")

def v_rotacion(lat):
    return v_ecuador * np.cos(np.radians(lat))

for lat in [0, 20, 40, 60, 80, 90]:
    print(f"latitud {lat:2d}°: {v_rotacion(lat):6.1f} m/s = {ms_a_kmh(v_rotacion(lat)):6.0f} km/h")

**Ejercicio.** Calcula la velocidad con la que la Tierra recorre su órbita alrededor del Sol, suponiendo que es circular de radio 1 UA = 1,496 × 10⁸ km y que da una vuelta en 365,256 días. Compáralo con el valor tabulado de 29,78 km/s.

In [ ]:
# Tu código aquí

In [ ]:
# Solución
UA = 1.496e8                 # km
T_orbita = 365.256 * 86400   # s
v_orbita = 2 * np.pi * UA / T_orbita
print(f"{v_orbita:.2f} km/s")

## 3. Movimiento uniforme y uniformemente acelerado

Con velocidad constante: $x(t) = x_0 + v\,t$. Con aceleración constante $a$:

$$v(t) = v_0 + a\,t, \qquad x(t) = x_0 + v_0\,t + \tfrac{1}{2}\,a\,t^2.$$

In [ ]:
def x_uniforme(t, x0, v):
    return x0 + v * t

def x_acelerado(t, x0, v0, a):
    return x0 + v0 * t + 0.5 * a * t**2

def v_acelerado(t, v0, a):
    return v0 + a * t

t = np.linspace(0, 10, 200)
fig, axs = plt.subplots(1, 2, figsize=(10, 3.5))
axs[0].plot(t, x_uniforme(t, 0, 3), label="uniforme, v = 3 m/s")
axs[0].plot(t, x_acelerado(t, 0, 0, 0.6), label="acelerado, a = 0,6 m/s²")
axs[0].set_xlabel("tiempo (s)"); axs[0].set_ylabel("posición (m)"); axs[0].legend(); axs[0].grid(alpha=0.3)
axs[1].plot(t, 3 + 0 * t, label="uniforme")
axs[1].plot(t, v_acelerado(t, 0, 0.6), label="acelerado")
axs[1].set_xlabel("tiempo (s)"); axs[1].set_ylabel("velocidad (m/s)"); axs[1].legend(); axs[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

La pendiente de la gráfica de posición es la velocidad, y la pendiente de la de velocidad es la aceleración. Es el germen de la derivada, que verás en la semana 10.

## 4. Simular el movimiento paso a paso

En lugar de usar la fórmula, podemos hacer avanzar el reloj un intervalo pequeño $\Delta t$ cada vez: actualizamos la velocidad con la aceleración y la posición con la velocidad. Es lo que hacen todas las simulaciones físicas.

In [ ]:
def simula(x0, v0, a, t_final, dt):
    """Simula un movimiento con aceleración constante avanzando el reloj de dt en dt."""
    t, x, v = 0.0, x0, v0
    xs, ts = [x], [t]
    while t < t_final - 1e-12:
        x = x + v * dt           # la posición cambia con la velocidad actual
        v = v + a * dt           # la velocidad cambia con la aceleración
        t = t + dt
        xs.append(x); ts.append(t)
    return np.array(ts), np.array(xs)

ts, xs = simula(0, 0, 0.6, 10, 0.5)
print(f"Paso a paso (dt = 0,5 s): x(10) = {xs[-1]:.2f} m")
print(f"Fórmula:                  x(10) = {x_acelerado(10, 0, 0, 0.6):.2f} m")

Hay una diferencia. Reduce `dt` y comprueba cómo desaparece.

In [ ]:
for dt in [1, 0.5, 0.1, 0.01, 0.001]:
    ts, xs = simula(0, 0, 0.6, 10, dt)
    print(f"dt = {dt:6.3f} s: x(10) = {xs[-1]:7.3f} m   error = {xs[-1] - x_acelerado(10, 0, 0, 0.6):+.3f} m")

## 5. Para pensar

1. ¿A qué velocidad te mueves por la rotación de la Tierra desde tu latitud? ¿Y respecto al Sol, por la órbita? ¿Qué quiere decir que "estás parado"?
2. ¿Cómo cambia el error de la simulación al dividir `dt` entre 10? ¿Qué relación guardan ambos?
3. Una piedra cae con $a = 9{,}80665$ m/s². Simula su caída durante 3 s y calcula cuánto ha bajado y a qué velocidad va.